In [2]:
'''
    20260113:
        设置合适的F1score的参数，获取最合适的方法！
'''

'\n    20260113:\n        设置合适的F1score的参数，获取最合适的方法！\n'

# import lib

In [47]:
import numpy as np
import datasets

import transformers

import json
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os
# download checkpoint
from accelerate import load_checkpoint_and_dispatch
from tqdm import tqdm

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # To prevent long warnings :)

#from accelerate import load_checkpoint_and_dispatch

from accelerate import init_empty_weights
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

#import util
#import importlib

#importlib.reload(langspecf1_utils)      # 只能 reload 模块本身
#from util import calc_ppl, get_test_data   # reload 后再重新 import 函数

import pandas as pd
import copy


## reload utils

In [48]:
#import util
import importlib
import langspecf1_utils
importlib.reload(langspecf1_utils)      # 只能 reload 模块本身
#from util import calc_ppl, get_test_

<module 'langspecf1_utils' from '/autodl-fs/data/LRP/langspecf1_utils.py'>

# 20260113

In [12]:
# llama2 7b chat
judege_path ='/root/autodl-fs/LRP/open_ended_data_generation/20251204_all_exp/1208_all_generation4judge.json_gpt4o.json'

# llama2 7b base
#judege_path ='/root/autodl-fs/LRP/open_ended_data_generation/20251210_llama2_7b_base/1210_all_generation4judge.json_gpt4o.json'

# llama2 7b base lang neuron & chat generation
#judege_path= '/root/autodl-fs/LRP/open_ended_data_generation/20251210_llama2_7b_neuronFromBase_GenerationFromChat/20251210_LRP_generation_all_models.json4judge.json_gpt4o.json'


# llama2 7b base LRP random mask
#judege_path= '/root/autodl-fs/LRP/open_ended_data_generation/20251218_LRP_random_mask/20251218_LRP_generation_all.json4judge.json_gpt4o.json'


In [11]:
'''
    get LAPE result

'''

# lape neuronFromBase & chat generation
ds_lape_mask_base = datasets.load_dataset('json', data_files = '/root/autodl-fs/LRP/open_ended_data_generation/20251204_all_exp/1208_all_generation4judge.json_gpt4o.json')




df_lape_mask_base = ds_lape_mask_base['train'].to_pandas()

df_lape_mask_base_en = df_lape_mask_base.loc[(df_lape_mask_base['model_type']=='mask_en|open_ended')].reset_index(drop=True)
df_lape_mask_base_vi = df_lape_mask_base.loc[(df_lape_mask_base['model_type']=='mask_vi|open_ended')].reset_index(drop=True)

df_lape_mask_base_zh = df_lape_mask_base.loc[(df_lape_mask_base['model_type']=='mask_zh|open_ended')].reset_index(drop=True)

df_lape_mask_lape = pd.concat([df_lape_mask_base_en,df_lape_mask_base_vi, df_lape_mask_base_zh ])


df_lape_mask_lape['model_type'].value_counts()



model_type
mask_en|open_ended    210
mask_vi|open_ended    210
mask_zh|open_ended    210
Name: count, dtype: int64

In [14]:
'''
    load LRP method result

'''

ds_judge_res = datasets.load_dataset('json', data_files = judege_path)

tmp = ds_judge_res['train'].to_pandas()
print(tmp['model_type'].value_counts())


model_type
mask_en|open_ended                                         210
mask_vi|open_ended                                         210
mask_zh|open_ended                                         210
gaprate_th_0.9_selected_LRP_kur_res_zh|open_ended          210
gaprate_th_11_selected_LRP_kur_res_en_zscore|open_ended    210
                                                          ... 
gaprate_th_0.99_selected_LRP_kur_res_vi|open_ended         210
gaprate_th_0.99_selected_LRP_kur_res_zh|open_ended         210
gaprate_th_0.9_selected_LRP_kur_res_en|open_ended          210
gaprate_th_0.9_selected_LRP_kur_res_vi|open_ended          210
org_model|open_ended                                       210
Name: count, Length: 154, dtype: int64


In [21]:
result_dict, df_judge_res = langspecf1_utils.get_analysis_res(ds_judge_res)

imodel_type: zh|org_model|open_ended
valid count: 70
imodel_type: en|mask_en|open_ended
valid count: 70
imodel_type: vi|mask_en|open_ended
valid count: 70
imodel_type: zh|mask_en|open_ended
valid count: 70
imodel_type: en|mask_vi|open_ended
valid count: 68
imodel_type: vi|mask_vi|open_ended
valid count: 70
imodel_type: zh|mask_vi|open_ended
valid count: 70
imodel_type: en|mask_zh|open_ended
valid count: 70
imodel_type: vi|mask_zh|open_ended
valid count: 69
imodel_type: zh|mask_zh|open_ended
valid count: 70
imodel_type: en|gaprate_th_0.9_selected_LRP_kur_res_zh|open_ended
valid count: 70
imodel_type: vi|gaprate_th_0.9_selected_LRP_kur_res_zh|open_ended
valid count: 70
imodel_type: zh|gaprate_th_0.9_selected_LRP_kur_res_zh|open_ended
valid count: 70
imodel_type: en|gaprate_th_11_selected_LRP_kur_res_en_zscore|open_ended
valid count: 70
imodel_type: zh|gap_rate_top_1perc_LRP_kur_res_vi|open_ended
valid count: 70
imodel_type: en|gap_rate_top_1perc_LRP_kur_res_zh|open_ended
valid count: 69


In [23]:
df_judge_res.head(1)

,text,lang,__index_level_0__,answer,model_type,judge_prompt,output,score,class_tpye
0,How can I improve my time management skills?,en,0,Time management is the process of planning and...,mask_en|open_ended,You are a neutral judge. Score the model’s ans...,"{""score"": 10, ""reason"": ""The answer is correct...",10,en|mask_en|open_ended


In [25]:
df_judge_res['mask_neuron_lang'] = df_judge_res['class_tpye'].apply(langspecf1_utils.get_mask_neuron_lang)
df_judge_res['method_name'] = df_judge_res['class_tpye'].apply(langspecf1_utils.get_method_name)

In [26]:
df_judge_res['method_name'].value_counts()

method_name
mask                                                                       630
gaprate_th_0.9_selected_LRP_kur_res                                        630
gaprate_th_11_selected_LRP_kur_res_zscore                                  630
gaprate_th_14_selected_LRP_kur_res_zscore                                  630
gaprate_th_42_selected_LRP_kur_res_zscore                                  630
gaprate_th_70_selected_LRP_kur_res_zscore                                  630
soft_preference_th_0.5_selected_LRP_kur_res_Not_zscore_margin_selected     630
soft_preference_th_0.5_selected_LRP_kur_res_zscore_margin_selected         630
soft_preference_th_0.95_selected_LRP_kur_res_Not_zscore_margin_selected    630
soft_preference_th_0.95_selected_LRP_kur_res_zscore_margin_selected        630
soft_preference_th_0.99_selected_LRP_kur_res_Not_zscore_margin_selected    630
soft_preference_th_0.99_selected_LRP_kur_res_zscore_margin_selected        630
soft_preference_th_0.9_selected_LRP_kur_

In [42]:
df_judge_res_no_mask = df_judge_res.loc[df_judge_res['method_name'] != 'mask'].reset_index(drop=True)

In [49]:
res_score, res_acuall_score, lang_score_baseline = langspecf1_utils.calc_metric(df_judge_res, result_dict)

In [50]:
lang_score_baseline

{'en': np.float64(7.9),
 'vi': np.float64(6.985714285714286),
 'zh': np.float64(6.814285714285714)}

In [53]:
# res_score_mean
final_score={}
for imethod in res_score:
    final_score[imethod] = sum(res_score[imethod].values())/len(res_score[imethod].values())

In [54]:
sorted(final_score.items(), key = lambda x: x[1], reverse=True )

[('th_0_selected_LRP_kur_res_zscore', np.float64(0.6125969526588811)),
 ('gaprate_th_0.9_selected_LRP_kur_res', np.float64(0.579540951087698)),
 ('soft_preference_th_0.5_selected_LRP_kur_res_Not_zscore_margin_selected',
  np.float64(0.5012965592106585)),
 ('gaprate_th_14_selected_LRP_kur_res_zscore', np.float64(0.4633640384915479)),
 ('soft_preference_th_0.5_selected_LRP_kur_res_zscore_margin_selected',
  np.float64(0.45318046205064544)),
 ('gaprate_th_11_selected_LRP_kur_res_zscore',
  np.float64(0.45115782709269747)),
 ('gaprate_th_0.95_selected_LRP_kur_res', np.float64(0.4447493977486747)),
 ('th_1_selected_LRP_kur_res_zscore', np.float64(0.34680091054863244)),
 ('gaprate_th_42_selected_LRP_kur_res_zscore',
  np.float64(0.32220121782860467)),
 ('gap_rate_bottom_1perc_LRP_kur_res', np.float64(0.32215455773317414)),
 ('soft_preference_th_0.9_selected_LRP_kur_res_Not_zscore_margin_selected',
  np.float64(0.31878238826747324)),
 ('top_1perc_LRP_kur_res', np.float64(0.31447038214879025))

In [57]:
'''
    show method result
'''
def show_result(mname):
    print('='*20)
    print('res_acuall_score:', str(res_acuall_score[mname]))
    print('='*20)
    print('langspecF1_score:', res_score[mname])



In [58]:
show_result('mask')

res_acuall_score: {'en': {'en': np.float64(7.942857142857143), 'vi': np.float64(7.128571428571429), 'zh': np.float64(6.957142857142857)}, 'vi': {'en': np.float64(7.685714285714286), 'vi': np.float64(7.3), 'zh': np.float64(6.9714285714285715)}, 'zh': {'en': np.float64(8.028571428571428), 'vi': np.float64(7.185714285714286), 'zh': np.float64(6.728571428571429)}}
langspecF1_score: {'en': np.float64(0.0), 'vi': np.float64(0.0), 'zh': np.float64(0.024844720496862583)}
